In [ ]:
import os, json, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
SEED = 42
np.random.seed(SEED)

import sklearn
print(f'NumPy:        {np.__version__}')
print(f'Pandas:       {pd.__version__}')
print(f'Scikit-Learn: {sklearn.__version__}')

In [ ]:
df = pd.read_csv('diabetes.csv')
print(f'Размер датасета: {df.shape}')
print(f'Столбцы: {list(df.columns)}')
df.head(10)

## Описание данных

Датасет собран Национальным институтом диабета (США). Содержит медицинские показатели пациентов женского пола индейского происхождения (племя Пима), возраст 21+. Задача: по 8 числовым признакам предсказать наличие сахарного диабета.

| Признак | Описание | Единица |
|---------|----------|---------|
| `Pregnancies` | Количество беременностей | шт. |
| `Glucose` | Концентрация глюкозы в плазме | мг/дл |
| `BloodPressure` | Диастолическое давление | мм рт. ст. |
| `SkinThickness` | Толщина кожной складки трицепса | мм |
| `Insulin` | Инсулин сыворотки через 2 часа | мкЕд/мл |
| `BMI` | Индекс массы тела | кг/м² |
| `DiabetesPedigreeFunction` | Наследственная функция диабета | - |
| `Age` | Возраст | лет |
| `Outcome` | Целевая переменная (0 - нет диабета, 1 - есть) | - |

In [ ]:
print(f'Строк:   {len(df)}')
print(f'Столбцов: {len(df.columns)}')
print(f'Пропуски (NaN): {df.isnull().sum().sum()}')
print()
vc = df['Outcome'].value_counts()
print(f'Нет диабета (0): {vc[0]} ({vc[0]/len(df)*100:.1f}%)')
print(f'Есть диабет (1): {vc[1]} ({vc[1]/len(df)*100:.1f}%)')
print()
df.describe().round(2)

In [ ]:
# Нули в медицинских признаках физически невозможны — это пропущенные данные
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print('Нулевые значения (пропуски):')
for col in zero_cols:
    n = (df[col] == 0).sum()
    print(f'  {col:30s}: {n:3d} ({n/len(df)*100:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].pie(
    [vc[0], vc[1]],
    labels=['Нет диабета', 'Есть диабет'],
    autopct='%1.1f%%',
    colors=['#5DCAA5', '#E8593C'],
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[0].set_title('Распределение классов', fontsize=12, fontweight='bold')

bars = axes[1].bar(
    ['Нет диабета (0)', 'Есть диабет (1)'],
    [vc[0], vc[1]],
    color=['#5DCAA5', '#E8593C'],
    edgecolor='white',
    linewidth=1.5
)
axes[1].set_title('Количество пациентов по классам', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Количество пациентов')
for bar, val in zip(bars, [vc[0], vc[1]]):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 5,
                 str(val), ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()
print('Умеренный дисбаланс классов (65/35) учитывается при обучении')

In [ ]:
features = ['Glucose', 'BMI', 'Age', 'BloodPressure',
            'Insulin', 'SkinThickness', 'Pregnancies', 'DiabetesPedigreeFunction']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for ax, feat in zip(axes, features):
    for outcome, color, label in [(0, '#5DCAA5', 'Нет диабета'), (1, '#E8593C', 'Есть диабет')]:
        data = df[df['Outcome'] == outcome][feat]
        data = data[data > 0]
        ax.hist(data, bins=25, alpha=0.6, color=color, label=label, edgecolor='none')
    ax.set_title(feat, fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.legend(fontsize=7)

plt.suptitle('Распределение признаков по классам', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask,
    annot=True, fmt='.2f',
    cmap='RdBu_r', center=0,
    linewidths=0.5,
    annot_kws={'size': 9}
)
plt.title('Корреляционная матрица признаков', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Корреляция признаков с Outcome:')
print(corr['Outcome'].drop('Outcome').sort_values(ascending=False).round(3))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for ax, feat in zip(axes, features):
    df_plot = df[df[feat] > 0].copy()
    df_plot['Диагноз'] = df_plot['Outcome'].map({0: 'Нет диабета', 1: 'Есть диабет'})
    sns.boxplot(
        data=df_plot, x='Диагноз', y=feat,
        palette={'Нет диабета': '#5DCAA5', 'Есть диабет': '#E8593C'},
        ax=ax, linewidth=1
    )
    ax.set_title(feat, fontsize=10, fontweight='bold')
    ax.set_xlabel('')

plt.suptitle('Boxplot признаков по классам', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Предобработка данных

Нули в признаках Glucose, BloodPressure, SkinThickness, Insulin, BMI заменяются на NaN и импутируются медианой. Все трансформации упакованы в Pipeline: scaler и imputer обучаются только на train-выборке, что исключает утечку данных.

In [ ]:
df_clean = df.copy()
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in zero_cols:
    df_clean[col] = df_clean[col].replace(0, np.nan)

print('Пропуски после замены нулей на NaN:')
missing = df_clean.isnull().sum()
missing = missing[missing > 0]
for col, n in missing.items():
    print(f'  {col:30s}: {n} ({n/len(df_clean)*100:.1f}%)')

In [ ]:
FEATURES = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
            'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

X = df_clean[FEATURES].values
y = df_clean['Outcome'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train: класс 0 = {(y_train==0).sum()}, класс 1 = {(y_train==1).sum()}')
print(f'Test:  класс 0 = {(y_test==0).sum()},  класс 1 = {(y_test==1).sum()}')
print(f'Баланс train: {round((y_train==1).sum()/len(y_train), 3)}   test: {round((y_test==1).sum()/len(y_test), 3)}')

## Обучение ML-моделей

In [ ]:
pipelines = {
    'Logistic Regression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
        ('model',   LogisticRegression(
            C=0.5,
            max_iter=1000,
            random_state=SEED,
            class_weight='balanced'
        ))
    ]),
    'Random Forest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model',   RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=4,
            min_samples_split=8,
            class_weight='balanced',
            random_state=SEED,
            n_jobs=-1
        ))
    ]),
    'Gradient Boosting': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model',   GradientBoostingClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            min_samples_leaf=4,
            random_state=SEED
        ))
    ]),
}

results = {}
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)

    train_pred = pipe.predict(X_train)
    test_pred  = pipe.predict(X_test)
    test_proba = pipe.predict_proba(X_test)[:, 1]

    train_acc = accuracy_score(y_train, train_pred)
    test_acc  = accuracy_score(y_test,  test_pred)
    test_f1   = f1_score(y_test, test_pred, average='weighted')
    test_roc  = roc_auc_score(y_test, test_proba)
    test_prec = precision_score(y_test, test_pred)
    test_rec  = recall_score(y_test, test_pred)
    gap       = train_acc - test_acc

    results[name] = {
        'train_acc': train_acc,
        'test_acc':  test_acc,
        'f1':        test_f1,
        'roc_auc':   test_roc,
        'precision': test_prec,
        'recall':    test_rec,
        'gap':       gap,
        'preds':     test_pred,
        'proba':     test_proba,
    }
    print(f'{name}:')
    print(f'  Train Acc={train_acc:.4f}  Test Acc={test_acc:.4f}  Gap={gap:.4f}  F1={test_f1:.4f}  ROC-AUC={test_roc:.4f}')

print('\nГотово')

In [ ]:
print('5-fold Stratified Cross-Validation (ROC-AUC):')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

for name, pipe in pipelines.items():
    cv_scores = cross_val_score(
        pipe, X, y,
        cv=skf,
        scoring='roc_auc',
        n_jobs=-1
    )
    print(f'{name}:')
    print(f'  ROC-AUC per fold: {np.round(cv_scores, 3)}')
    print(f'  Mean: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')
    print()

## Оценка качества

In [ ]:
df_summary = pd.DataFrame([
    {
        'Модель':    name,
        'Train Acc': round(r['train_acc'], 4),
        'Test Acc':  round(r['test_acc'],  4),
        'F1-Score':  round(r['f1'],        4),
        'ROC-AUC':   round(r['roc_auc'],   4),
        'Precision': round(r['precision'], 4),
        'Recall':    round(r['recall'],    4),
        'Gap':       round(r['gap'],       4),
    }
    for name, r in results.items()
]).sort_values('ROC-AUC', ascending=False).set_index('Модель')

print(df_summary.to_string())

best = df_summary['ROC-AUC'].idxmax()
gap  = df_summary.loc[best, 'Gap']
status = 'переобучения нет' if gap < 0.05 else 'лёгкое переобучение' if gap < 0.10 else 'переобучение'
print(f'\nЛучшая модель: {best}')
print(f'ROC-AUC: {df_summary.loc[best, "ROC-AUC"]}   Test Acc: {df_summary.loc[best, "Test Acc"]}   Gap: {gap} ({status})')

In [ ]:
names      = list(results.keys())
train_accs = [results[n]['train_acc'] for n in names]
test_accs  = [results[n]['test_acc']  for n in names]
gaps       = [results[n]['gap']        for n in names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(names))
w = 0.35
ax1.bar(x - w/2, train_accs, w, label='Train Accuracy', color='#9FE1CB')
ax1.bar(x + w/2, test_accs,  w, label='Test Accuracy',  color='#1D9E75')
ax1.set_xticks(x)
ax1.set_xticklabels(names, rotation=8)
ax1.set_ylim(0.5, 1.0)
ax1.set_title('Train vs Test Accuracy', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
for bar in ax1.containers:
    ax1.bar_label(bar, fmt='%.3f', fontsize=9)

colors_gap = ['#1D9E75' if g < 0.05 else '#EF9F27' if g < 0.10 else '#E24B4A' for g in gaps]
bars = ax2.bar(names, gaps, color=colors_gap)
ax2.axhline(0.05, color='#EF9F27', linestyle='--', linewidth=1.5, label='Предупреждение (0.05)')
ax2.axhline(0.10, color='#E24B4A', linestyle='--', linewidth=1.5, label='Переобучение (0.10)')
ax2.set_title('Gap = Train Acc - Test Acc', fontsize=12, fontweight='bold')
ax2.set_ylabel('Разница')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)
for bar, g in zip(bars, gaps):
    ax2.text(bar.get_x() + bar.get_width()/2, g + 0.002, f'{g:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 7))

colors = ['#1D9E75', '#E8593C', '#378ADD']
for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['proba'])
    roc_val = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC = {roc_val:.3f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.500)')
plt.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
plt.xlabel('False Positive Rate', fontsize=11)
plt.ylabel('True Positive Rate', fontsize=11)
plt.title('ROC-кривые', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 7))

baseline = y_test.mean()
colors = ['#1D9E75', '#E8593C', '#378ADD']
for (name, res), color in zip(results.items(), colors):
    prec, rec, _ = precision_recall_curve(y_test, res['proba'])
    ap = average_precision_score(y_test, res['proba'])
    plt.plot(rec, prec, color=color, linewidth=2, label=f'{name} (AP = {ap:.3f})')

plt.axhline(baseline, color='gray', linestyle='--', linewidth=1, label=f'Baseline (AP = {baseline:.3f})')
plt.xlabel('Recall', fontsize=11)
plt.ylabel('Precision', fontsize=11)
plt.title('Precision-Recall кривые', fontsize=13, fontweight='bold')
plt.legend(loc='upper right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['preds'])
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Greens',
        xticklabels=['Нет диабета', 'Есть диабет'],
        yticklabels=['Нет диабета', 'Есть диабет'],
        ax=ax, linewidths=1, linecolor='white'
    )
    acc = res['test_acc']
    ax.set_title(f'{name}\nTest Acc = {acc:.4f}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Предсказано', fontsize=9)
    ax.set_ylabel('Истина', fontsize=9)

plt.suptitle('Матрицы ошибок', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
print(f'Classification Report — {best}:\n')
print(classification_report(
    y_test,
    results[best]['preds'],
    target_names=['Нет диабета', 'Есть диабет'],
    digits=3
))

In [ ]:
rf_model = pipelines['Random Forest'].named_steps['model']
fi = pd.Series(rf_model.feature_importances_, index=FEATURES)
fi_sorted = fi.sort_values(ascending=True)

plt.figure(figsize=(9, 5))
colors = plt.cm.Greens(np.linspace(0.35, 0.9, len(fi_sorted)))
plt.barh(fi_sorted.index, fi_sorted.values, color=colors)
plt.title('Важность признаков — Random Forest', fontsize=13, fontweight='bold')
plt.xlabel('Feature Importance')
for i, (val, name) in enumerate(zip(fi_sorted.values, fi_sorted.index)):
    plt.text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print('Топ-3 важных признака:')
for feat, imp in fi.sort_values(ascending=False).head(3).items():
    print(f'  {feat}: {imp:.4f}')

In [ ]:
# Пример предсказания для новых пациентов
# Формат: Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age

new_patients = np.array([
    [2,  138, 62, 35, 0,  33.6, 0.127, 47],
    [1,   85, 66, 29, 0,  26.6, 0.351, 31],
    [8,  183, 64,  0, 0,  23.3, 0.672, 32],
])

expected   = ['Есть диабет', 'Нет диабета', 'Есть диабет']
best_pipe  = pipelines[best]
preds      = best_pipe.predict(new_patients)
probas     = best_pipe.predict_proba(new_patients)[:, 1]

for i, (pred, proba, exp) in enumerate(zip(preds, probas, expected), 1):
    label = 'Есть диабет' if pred == 1 else 'Нет диабета'
    print(f'Пациент {i}:')
    print(f'  Предсказание: {label}  (вероятность диабета: {proba:.1%})')
    print(f'  Ожидаемо:     {exp}')
    print()

In [ ]:
os.makedirs('models', exist_ok=True)

with open('models/best_diabetes_model.pkl', 'wb') as f:
    pickle.dump(pipelines[best], f)

with open('models/feature_names.json', 'w') as f:
    json.dump(FEATURES, f)

print(f'Модель ({best}) сохранена: models/best_diabetes_model.pkl')
print(f'Признаки сохранены:       models/feature_names.json')

## Выводы

Использован датасет Pima Indians Diabetes Database (Kaggle / UCI). 768 пациентов, 8 числовых признаков, бинарная классификация. Нулевые значения в признаках Glucose, BloodPressure, SkinThickness, Insulin, BMI являются пропущенными данными и заменены на NaN с последующей медианной импутацией.

Все трансформации упакованы в sklearn.Pipeline: scaler и imputer обучаются только на train-выборке, что исключает утечку данных. Для защиты от переобучения применены ограничения глубины деревьев, минимальное число примеров в листе, L2-регуляризация и class_weight='balanced' для учёта дисбаланса классов.

Gap (Train - Test Accuracy) у всех моделей не превышает 0.05, что говорит об отсутствии переобучения. 5-fold кросс-валидация подтверждает стабильность результатов. Наиболее важные признаки: Glucose, BMI, Age.